In [1]:
import pandas as pd
import duckdb as ddb
from pathlib import Path

In [2]:
DATAPATH = '/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet'

In [3]:
con = ddb.connect(':memory:')

In [4]:
# importing the raw node data
# read parquet
con.execute(f"""CREATE TABLE nodes_table AS SELECT * FROM read_parquet('{DATAPATH}')""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
con.query("""SELECT * FROM nodes_table LIMIT 5""").show()

┌───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬───────┬─────────────┬────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────┬─────────────┬──────────────┬──────────────┬───────────┬───────────┬────────────┬────────────┬───────────────────────┬───────────────────────┬──────────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┬──────────────────────────┬───────────┬────────────┬─────────────────────────┬─────────────────────────────────┐
│ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_percent │ cpu_interrupt_percent │ load_shortterm_percent │ load_midterm_percent │ load_longter

In [6]:
VM_HARDWARE = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/vms/2024-12-14T000000Z_2025-04-13T235959Z/vms.csv"
VM_DATA = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/**/*.csv"

## VM HARDWARE

In [7]:
# VM HARDWARE 
con.query(f"""CREATE OR REPLACE TABLE vmhardware AS SELECT * FROM '{VM_HARDWARE}'""")
con.query("""SELECT * FROM vmhardware LIMIT 5""").show()

┌──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┐
│  vm_id   │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │
│ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │
├──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┤
│ 6239bcf4 │ 39786247 │ 4ba6ce42   │ 85ecdff4  │   32.0 │   60000.0 │    30.0 │        160.0 │
│ 80458243 │ bd41a7e4 │ 74dfdf00   │ cb92696e  │    1.0 │    2000.0 │    10.0 │         20.0 │
│ 5d9f6237 │ 7190ddfa │ 906147a9   │ 78c5906a  │    4.0 │    7500.0 │    30.0 │         60.0 │
│ 941785dc │ 1a324c91 │ f09054da   │ 23658854  │    2.0 │    4000.0 │    10.0 │        100.0 │
│ fb856e0f │ c1eb1c1c │ f09054da   │ c015d680  │    4.0 │    7500.0 │    30.0 │         60.0 │
└──────────┴──────────┴────────────┴───────────┴────────┴───────────┴─────────┴──────────────┘



## VM DATA

In [8]:
con.execute(f"""
                CREATE OR REPLACE VIEW vm_data AS
                SELECT *
                FROM read_csv_auto('{VM_DATA}',
                                filename=false,
                                union_by_name=true)
            """)

### Merging

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE vm_merged AS
    SELECT 
        d.*, 
        h.*
    FROM vm_data d
    LEFT JOIN vmhardware h
    ON d.vm_id = h.vm_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Imputing missing values
Forward Fill and 0s

In [10]:
con.execute("""
    CREATE OR REPLACE TABLE vm_filled AS
    SELECT *,
        LAST_VALUE(scaphandre_vm_power_total_watts IGNORE NULLS)
        OVER (
            PARTITION BY vm_id 
            ORDER BY timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS power_filled
    FROM vm_merged
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE vm_final AS
    SELECT *,
        COALESCE(power_filled, 0) AS power_clean
    FROM vm_filled
 """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.query("""SELECT * FROM vm_final LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┬──────────────┬─────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ vm_id_1  │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │ power_filled │ power_clean │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │    double    │   double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┼──────────────┼─────────────┤
│ 2025-02-06 18:27:00+01   │ c3cc7b6e │ e4362c19        │ a6177608  

In [13]:
#TIMESTAMP = '2025-01-16T08:33:00Z'
TIMESTAMP = '2025-01-16 08:33:00 UTC'

### Get VM list per node 

In [24]:
# Check timestamp notation
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        LIST(vm_id) AS vm_ids
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬──────────────────────────────────────────────────────────────────────────────────┐
│ node_name │                                      vm_ids                                      │
│  varchar  │                                    varchar[]                                     │
├───────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 7c804642  │ [cb521e0f]                                                                       │
│ b654db47  │ [e80b91dd]                                                                       │
│ 0eb86326  │ [ef84013a]                                                                       │
│ c9900ab9  │ [5486492a, 654a92dd]                                                             │
│ 6bb5c698  │ [f545ef7a, 4aeb93ea, be1a8732, b963d466, eae4dca6]                               │
│ ea7f6999  │ [91dc05b7]                                                                       │
│ 6d834fb8  │ [c6b13b89]      

### VM Aggregation - get node load - use for comparison - ALSO nodes_t

In [16]:
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        SUM(power_clean) AS total_power,
        COUNT(*) AS vm_count
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬─────────────────────┬──────────┐
│ node_name │     total_power     │ vm_count │
│  varchar  │       double        │  int64   │
├───────────┼─────────────────────┼──────────┤
│ 82415d68  │                0.44 │       11 │
│ d8d6d1e0  │                0.11 │        1 │
│ 6342c79e  │                0.18 │        1 │
│ 7f4f3762  │                0.12 │        6 │
│ 28e8b849  │                0.01 │        1 │
│ 3d7960e7  │  1.1600000000000001 │        7 │
│ 58512373  │                0.25 │        7 │
│ 5c5e400b  │                0.07 │        1 │
│ 5333662e  │              118.55 │        1 │
│ e2b6ed0e  │   9.450000000000001 │        5 │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│ a22bdd7c  │                0.07 │        2 │
│ 7f384201  │ 0.22000000000000003 │        4 │
│ 6ff55332  │                0.24 │        3 │
│ 8d5a6c7e  │                0.41 │        1 │
│ 764b2e98  │

# Timestamps

In [17]:
timestamps = con.execute("""
    SELECT DISTINCT timestamp AT TIME ZONE 'UTC'
    FROM vm_final
    ORDER BY timestamp
""").fetchall()

In [14]:
LOWER_THRESHOLD = 10
UPPER_THRESHOLD = 85

In [45]:
def sortDecreasingUtilization(vmList, total_vm_power):

    # Input: List of vm_ids and corresponding power usage, and total vm power for the node
    # Output: List of vm_ids sorted by decreasing load (as a percentage of total vm power)
    vmList['vm_load'] = vmList['vm_power'] / total_vm_power * 100
    sorted_vm_df = vmList.sort_values(by='vm_load', ascending=False)
    return sorted_vm_df['vm_id'].to_list(), sorted_vm_df['vm_power'].to_list(), sorted_vm_df['vm_load'].to_list()

In [41]:
 # for each node at time t, extract name, cpu usage, and vm count and vm ids
node_df_t = con.execute("""
                        SELECT 
                            n.node_name,
                            n.cpu_usage_percent,
                            n.ipmi_system_power_watts, 
                            COUNT(v.vm_id) AS vm_count,
                            LIST(v.vm_id) AS vm_ids,
                            LIST(v.power_clean) AS vm_powers,
                            SUM(v.power_clean) AS total_vm_power
                        FROM nodes_table n
                        LEFT JOIN vm_final v
                            ON n.timestamp = v.timestamp
                            AND n.node_name = v.hypervisor_name
                        WHERE n.timestamp = ?
                        GROUP BY n.node_name, n.cpu_usage_percent, n.ipmi_system_power_watts
                    """, [TIMESTAMP]).df()

    
overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]


In [ ]:
overloaded['vm_loads'] = None
underloaded['vm_loads'] = None

for idx, row in overloaded.iterrows():
    # replace the original vm_ids and vm_powers with the sorted ones (and calculate vm load)
    vm_ids, vm_powers, vm_loads = sortDecreasingUtilization(
        pd.DataFrame({
            'vm_id': row['vm_ids'],
            'vm_power': row['vm_powers']
        }),
        row['total_vm_power']
    )

    overloaded.at[idx, 'vm_ids'] = vm_ids
    overloaded.at[idx, 'vm_powers'] = vm_powers
    overloaded.at[idx, 'vm_loads'] = vm_loads

for idx, row in underloaded.iterrows():
    # replace the original vm_ids and vm_powers with the sorted ones (and calculate vm load)
    vm_ids, vm_powers, vm_loads = sortDecreasingUtilization(
        pd.DataFrame({
            'vm_id': row['vm_ids'],
            'vm_power': row['vm_powers']
        }),
        row['total_vm_power']
    )

    underloaded.at[idx, 'vm_ids'] = vm_ids
    underloaded.at[idx, 'vm_powers'] = vm_powers
    underloaded.at[idx, 'vm_loads'] = vm_loads

In [43]:
overloaded

,node_name,cpu_usage_percent,ipmi_system_power_watts,vm_count,vm_ids,vm_powers,total_vm_power,vm_loads
27,05c5ef00,99.65,784.17,1,[06e83870],[521.42],521.42,[100.0]
32,46de937b,91.68,785.00,0,[None],[nan],NaN,[nan]
37,5433c3e6,99.95,780.83,1,[b3382983],[492.27],492.27,[100.0]
44,764b2e98,100.00,800.00,1,[525134f6],[525.23],525.23,[100.0]
59,2eea5215,85.99,785.00,0,[None],[nan],NaN,[nan]
74,88b02143,86.38,786.67,1,[857a76fa],[454.25],454.25,[100.0]
88,cbf4a974,99.92,788.33,1,[5d14c1f6],[524.4],524.40,[100.0]


In [44]:
underloaded

,node_name,cpu_usage_percent,ipmi_system_power_watts,vm_count,vm_ids,vm_powers,total_vm_power,vm_loads
1,5d7db037,0.16,415.00,7,"[95b66945, ef9152aa, 32d947ff, af20ec04, 0c898...","[0.1, 0.06, 0.03, 0.03, 0.02, 0.02, 0.0]",0.26,"[38.46153846153847, 23.076923076923077, 11.538..."
2,e41df102,0.32,416.67,5,"[ac0f1f33, fb856e0f, c9afd608, 6fcacc4e, 29919...","[0.43, 0.29, 0.03, 0.02, 0.01]",0.78,"[55.128205128205124, 37.179487179487175, 3.846..."
3,a639d055,0.81,278.33,1,[fe6a63da],[0.2],0.20,[100.0]
4,54f5286d,0.93,77.33,1,[465cef03],[0.5],0.50,[100.0]
6,a22bdd7c,0.07,407.50,2,"[b2f57d95, dd8ec43e]","[0.04, 0.03]",0.07,"[57.14285714285714, 42.85714285714285]"
...,...,...,...,...,...,...,...,...
122,d35334d1,0.34,283.33,1,[8a4874ee],[0.19],0.19,[100.0]
123,79b5307d,1.22,76.67,1,[15497bd1],[0.54],0.54,[100.0]
124,366801,0.62,412.50,0,[None],[nan],NaN,[nan]
125,a2ffa1e3,0.09,61.67,0,[None],[nan],NaN,[nan]


There are mony occurences in which the node is being underutilised but is not 0 and there are no VMs on them. 
This does not affect host detection but when it comes to vm selection there are no VMs to select and non to migrate so the node should be put to idle

In [ ]:
# Example


In [ ]:
def minimization_of_migrations(vms, loads, cpu_util, vms_to_migrate):
    best_fit_util = float('inf')
    while cpu_util > UPPER_THRESHOLD and vms:
        for i in range(len(vms)):
            if loads[i] > cpu_util - UPPER_THRESHOLD:
                util = loads[i] - cpu_util + UPPER_THRESHOLD
                print("NEW UTL:", util)
                if util < best_fit_util:
                    best_fit_util = util
                    best_fit_vm = vms[i]
        cpu_util -= best_fit_util
        print("CPU UTIL:", cpu_util)

        vms_to_migrate.append({
            "vm_id": best_fit_vm,
            "source_node": row["node_name"]
        })
        # remove the migrated vm from the list of vms and loads
        vms.remove(best_fit_vm)
        loads.remove(index=i)

In [ ]:
'''
    For underloaded nodes, we migrate all VMs to other nodes
    For overloaded nodes, we migrate VMs until we are under the upper threshold
'''
def selection_algorithm(overloaded, underloaded, t):
    vms_to_migrate = []

    # Underloaded nodes
    for _, row in underloaded.iterrows():
        if row["vm_count"] == 0:
            continue
        
        for vm_id in row["vm_ids"]:
            vms_to_migrate.append({
            "vm_id": vm_id,
            "source_node": row["node_name"]
            })

    # Overloaded nodes
    for _, row in overloaded.iterrows():
        if row["vm_count"] == 0:
            continue
        
        vm_ids = row['vm_ids']
        vm_loads = row['vm_loads']
        host_util = row["cpu_usage_percent"]
        # Minimization of migrations policy

        if len(vm_ids) > 1:
            vms_to_migrate = minimization_of_migrations(vm_ids, vm_loads, host_util, vms_to_migrate)
        else:
            vms_to_migrate.append({
                "vm_id": vm_ids[0],
                "source_node": row["node_name"]
            })

    return vms_to_migrate

In [ ]:
for (t,) in timestamps: 
    
    # for each node at time t, extract name, cpu usage, and vm count and vm ids
    node_df_t = con.execute("""
                            SELECT 
                                n.node_name,
                                n.cpu_usage_percent,
                                COUNT(v.vm_id) AS vm_count,
                                LIST(v.vm_id) AS vm_ids
                            FROM nodes_table n
                            LEFT JOIN vm_final v
                                ON n.timestamp = v.timestamp
                                AND n.node_name = v.hypervisor_name
                            WHERE n.timestamp = ?
                            GROUP BY n.node_name, n.cpu_usage_percent
                        """, [t + " UTC"]).df()
    
    # Host Detection
    overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
    underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]

    # VM Selection

    

In [ ]:
# Check if nodes with no vms have cpu usage to 0 -> they don't

In [ ]:
# total power consumption of the cluster at time t


In [3]:
import duckdb as ddb
from pathlib import Path

con = ddb.connect(':memory:')

DATA_DIR = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption"
con.execute(f"""CREATE OR REPLACE VIEW node_snapshot AS SELECT * FROM read_parquet('{DATA_DIR}/processed/node_snapshot.parquet')""")
con.query("""SELECT * FROM node_snapshot WHERE vm_count > 0 LIMIT 5""").show()

┌──────────────────────────┬───────────┬───────────────────┬─────────────────────────┬─────────────┬──────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────┐
│        timestamp         │ node_name │ cpu_usage_percent │ ipmi_system_power_watts │ host_state  │ vm_count │                                                                                             vm_ids                                                                                             │                                                   vm_powers                                                   │   total_vm_power   │
│ timestamp with time zone │  varchar  │      double       │         double          │   varchar   │  int64   │             

In [6]:
NODES_FEATURES = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet"
con.execute(f"""CREATE OR REPLACE TABLE nodes_table AS SELECT * FROM read_parquet('{NODES_FEATURES}')""")
con.query(f"""SELECT * from nodes_table LIMIT 5""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬───────┬─────────────┬────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────┬─────────────┬──────────────┬──────────────┬───────────┬───────────┬────────────┬────────────┬───────────────────────┬───────────────────────┬──────────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┬──────────────────────────┬───────────┬────────────┬─────────────────────────┬─────────────────────────────────┐
│ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_percent │ cpu_interrupt_percent │ load_shortterm_percent │ load_midterm_percent │ load_longter

In [5]:
VM_FINAL = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/processed/vm_final.parquet"
con.execute(f"""CREATE OR REPLACE TABLE vm_final AS SELECT * FROM read_parquet('{VM_FINAL}')""")
con.query(f"""SELECT * from vm_final LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬────────────┬────────┬───────────┬─────────┬──────────────┬──────────────┬─────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ user_id  │ project_id │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │ power_filled │ power_clean │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │  varchar   │ double │  double   │ double  │    double    │    double    │   double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼────────────┼────────┼───────────┼─────────┼──────────────┼──────────────┼─────────────┤
│ 2025-03-31 22:00:00+02   │ 3c92802c │ 85c6cd05        │ a6177608         │                            0.03 │ a16f591f │ 7d4a4a5a   │   32.0 │   92000.0 │    5

In [3]:
NODES_OPERATIONAL_FEATURES = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/nodes_operational_features.parquet"
con.execute(f"""CREATE OR REPLACE TABLE nodes_feat AS SELECT * FROM read_parquet('{NODES_OPERATIONAL_FEATURES}')""")
con.query(f"""SELECT * from nodes_feat LIMIT 5""").show()

┌───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┬──────────────────────────┬───────────┬────────────┬───────────────────┬─────────────────────────┬─────────────────────────────────┬──────────────────────────────┬──────────────────────────────────────┐
│ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_percent │ cpu_interrupt_percent │ load_shortterm_percent │ load_midterm_percent │ load_longterm_percent │  memory_util_ratio  │   disk_util_ratio   │ processes_util_ratio  │ processes_blocked_ratio │ total_threads │ total_cores │ rated_power_usable │ cpu_freq_ghz │ has_gpu │        timestamp         │ n

## Checking Node Snapshot 

In [20]:
import duckdb as ddb
from pathlib import Path

con = ddb.connect(':memory:')

In [21]:
con.execute(f"""CREATE OR REPLACE VIEW node_snapshot AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/node_snapshot.parquet')""")
con.execute(f"""CREATE OR REPLACE VIEW simulation_snapshot AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/simulated_consolidation.parquet')""")


In [26]:
TIMESTAMP = '2025-01-16 08:33:00'
t = TIMESTAMP
t = t + " UTC"
host_df = con.execute(f"SELECT * FROM node_snapshot WHERE timestamp = '{t}'").df()
simulation_df = con.execute(f"SELECT * FROM simulation_snapshot WHERE timestamp = '{t}'").df()

In [27]:
host_df.head()

,timestamp,node_name,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,host_state,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2025-01-16 09:33:00+01:00,0049db0c,0.18,415.83,128.0,1100.0,386653.667969,280158.358073,underloaded,0,0.00,0.0,0.0,415.83,415.83
1,2025-01-16 09:33:00+01:00,0064a367,23.35,497.50,128.0,1100.0,386653.667969,1608.076172,normal,1,82.98,128.0,370000.0,414.52,497.50
2,2025-01-16 09:33:00+01:00,0065ef1b,0.05,416.67,128.0,1100.0,386653.664062,364299.950521,underloaded,3,0.03,6.0,11500.0,416.64,416.67
3,2025-01-16 09:33:00+01:00,05c5ef00,99.65,784.17,128.0,1100.0,386653.667969,1312.126302,overloaded,1,521.42,128.0,370000.0,262.75,784.17
4,2025-01-16 09:33:00+01:00,0861be67,0.04,415.00,128.0,1100.0,386653.667969,374028.388672,underloaded,0,0.00,0.0,0.0,415.00,415.00


In [ ]:
con.execute(f"""CREATE OR REPLACE VIEW vm_final AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/vm_final.parquet')""")

In [23]:
vm_df = con.execute(f"""
    SELECT 
        hypervisor_name,
        vm_id, vm_cpu, vm_memory_mb, vm_power
    FROM vm_final
    WHERE timestamp = '{t}'
    ORDER BY hypervisor_name, vm_power DESC
""").df()

vms_by_host = {}
for host, group in vm_df.groupby('hypervisor_name'):
    vms_by_host[host] = group[['vm_id', 'vm_cpu', 'vm_memory_mb', 'vm_power']].to_dict('records')

# Now populate the host_df with this data
host_df['vm_ids'] = host_df['node_name'].map(lambda x: [v['vm_id'] for v in vms_by_host.get(x, [])])
host_df['vm_cpus'] = host_df['node_name'].map(lambda x: [v['vm_cpu'] for v in vms_by_host.get(x, [])])
host_df['vm_memories_mb'] = host_df['node_name'].map(lambda x: [v['vm_memory_mb'] for v in vms_by_host.get(x, [])])
host_df['vm_powers'] = host_df['node_name'].map(lambda x: [v['vm_power'] for v in vms_by_host.get(x, [])])

In [24]:
host_df.head()

,timestamp,node_name,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,host_state,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power,vm_ids,vm_cpus,vm_memories_mb,vm_powers
0,2025-01-16 09:33:00+01:00,0049db0c,0.18,415.83,128.0,1100.0,386653.667969,280158.358073,underloaded,0,0.00,0.0,0.0,415.83,415.83,[],[],[],[]
1,2025-01-16 09:33:00+01:00,0064a367,23.35,497.50,128.0,1100.0,386653.667969,1608.076172,normal,1,82.98,128.0,370000.0,414.52,497.50,[c3297a97],[128.0],[370000.0],[82.98]
2,2025-01-16 09:33:00+01:00,0065ef1b,0.05,416.67,128.0,1100.0,386653.664062,364299.950521,underloaded,3,0.03,6.0,11500.0,416.64,416.67,"[786824e6, 64ffb69d, da9f3d11]","[4.0, 1.0, 1.0]","[7500.0, 2000.0, 2000.0]","[0.03, 0.0, 0.0]"
3,2025-01-16 09:33:00+01:00,05c5ef00,99.65,784.17,128.0,1100.0,386653.667969,1312.126302,overloaded,1,521.42,128.0,370000.0,262.75,784.17,[06e83870],[128.0],[370000.0],[521.42]
4,2025-01-16 09:33:00+01:00,0861be67,0.04,415.00,128.0,1100.0,386653.667969,374028.388672,underloaded,0,0.00,0.0,0.0,415.00,415.00,[],[],[],[]


In [25]:
simulation_df.head()

,timestamp,node_name,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,host_state,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power,vm_ids,vm_cpus,vm_memories_mb,vm_powers
0,2025-01-16 09:33:00+01:00,0049db0c,0.18,415.83,128.0,1100.0,386653.667969,280158.358073,underloaded,0,0.00,0.0,0.0,415.83,0.00,[],[],[],[]
1,2025-01-16 09:33:00+01:00,0064a367,23.35,497.50,128.0,1100.0,386653.667969,1608.076172,normal,1,82.98,128.0,370000.0,414.52,497.50,[c3297a97],[128.0],[370000.0],[82.98]
2,2025-01-16 09:33:00+01:00,0065ef1b,0.05,416.67,128.0,1100.0,386653.664062,364299.950521,underloaded,3,0.03,6.0,11500.0,416.64,416.67,"[786824e6, da9f3d11, 64ffb69d]","[4.0, 1.0, 1.0]","[7500.0, 2000.0, 2000.0]","[0.03, 0.0, 0.0]"
3,2025-01-16 09:33:00+01:00,05c5ef00,99.65,784.17,128.0,1100.0,386653.667969,1312.126302,overloaded,1,521.42,128.0,370000.0,262.75,784.17,[06e83870],[128.0],[370000.0],[521.42]
4,2025-01-16 09:33:00+01:00,0861be67,0.04,415.00,128.0,1100.0,386653.667969,374028.388672,underloaded,0,0.00,0.0,0.0,415.00,0.00,[],[],[],[]


In [14]:
host_df.shape

(128, 18)

In [15]:
host_df.to_csv('example_host_df.csv', index=False)

## Total Energy Consumption

In [28]:
con.execute("""
    SELECT 
        'Original' as scenario,
        SUM(simulated_power) as total_power_w,
        AVG(simulated_power) as avg_power_w,
        COUNT(*) as host_count
    FROM node_snapshot
    UNION ALL
    SELECT 
        'Simulated' as scenario,
        SUM(simulated_power) as total_power_w,
        AVG(simulated_power) as avg_power_w,
        COUNT(*) as host_count
    FROM simulation_snapshot
""").df()

,scenario,total_power_w,avg_power_w,host_count
0,Original,2.184780e+09,321.190790,7070878
1,Simulated,1.312500e+09,190.132335,7068358


## Empty Hosts

In [29]:
con.execute("""
    SELECT 
        'Original' as scenario,
        COUNT(*) FILTER (WHERE vm_power_allocated <= 0) as empty_hosts,
        COUNT(*) as total_hosts
    FROM node_snapshot
    UNION ALL
    SELECT 
        'Simulated' as scenario,
        COUNT(*) FILTER (WHERE vm_power_allocated <= 0) as empty_hosts,
        COUNT(*) as total_hosts
    FROM simulation_snapshot
""").df()

,scenario,empty_hosts,total_hosts
0,Original,3035795,7070878
1,Simulated,3006954,7068358


## Avg Active hosts over time

In [35]:
con.execute("""
    SELECT 
        'Original' as scenario,
        AVG(active_hosts) as avg_active_hosts,
        MIN(active_hosts) as min_active_hosts,
        MAX(active_hosts) as max_active_hosts
    FROM (
        SELECT 
            timestamp,
            COUNT(*) FILTER (WHERE ipmi_system_power_watts > 0) as active_hosts
        FROM node_snapshot
        GROUP BY timestamp
    ) original_subquery
    UNION ALL
    SELECT 
        'Simulated' as scenario,
        AVG(active_hosts) as avg_active_hosts,
        MIN(active_hosts) as min_active_hosts,
        MAX(active_hosts) as max_active_hosts
    FROM (
        SELECT 
            timestamp,
            COUNT(*) FILTER (WHERE vm_power_allocated > 0) as active_hosts
        FROM simulation_snapshot
        GROUP BY timestamp
    ) simulated_subquery
""").df()

,scenario,avg_active_hosts,min_active_hosts,max_active_hosts
0,Original,117.116460,0,129
1,Simulated,69.975948,0,204


# 3. Energy savings over time

In [30]:
con.execute("""
    SELECT 
        n.timestamp,
        SUM(n.simulated_power) as original_power,
        SUM(s.simulated_power) as simulated_power,
        SUM(n.simulated_power) - SUM(s.simulated_power) as power_saved_w
    FROM node_snapshot n
    FULL OUTER JOIN simulation_snapshot s 
        ON n.timestamp = s.timestamp AND n.node_name = s.node_name
    GROUP BY n.timestamp
    ORDER BY n.timestamp
""").df()

,timestamp,original_power,simulated_power,power_saved_w
0,2024-12-14 01:00:00+01:00,44234.11,803.33,43430.78
1,2024-12-14 01:03:00+01:00,44431.64,798.33,43633.31
2,2024-12-14 01:06:00+01:00,44355.31,803.33,43551.98
3,2024-12-14 01:09:00+01:00,44428.48,803.33,43625.15
4,2024-12-14 01:12:00+01:00,44295.16,803.33,43491.83
...,...,...,...,...
58076,2025-04-14 01:48:00+02:00,38047.84,NaN,NaN
58077,2025-04-14 01:51:00+02:00,38054.68,NaN,NaN
58078,2025-04-14 01:54:00+02:00,37868.22,NaN,NaN
58079,2025-04-14 01:57:00+02:00,37846.04,NaN,NaN


# 4. Host-by-host comparison 

In [33]:
con.execute("""
    SELECT 
        n.node_name,
        n.cpu_allocated as orig_cpu,
        s.cpu_allocated as sim_cpu,
        n.memory_allocated_mb as orig_mem,
        s.memory_allocated_mb as sim_mem,
        n.simulated_power as orig_power,
        s.simulated_power as sim_power,
        s.simulated_power - n.simulated_power as power_delta
    FROM node_snapshot n
    FULL OUTER JOIN simulation_snapshot s
        ON n.node_name = s.node_name AND n.timestamp = s.timestamp
    WHERE n.timestamp = '2025-02-15 12:00:00'
    ORDER BY power_delta DESC
""").df()

,node_name,orig_cpu,sim_cpu,orig_mem,sim_mem,orig_power,sim_power,power_delta
0,1e64c783,52.0,128.0,96500.0,260500.0,600.00,619.51,19.51
1,e2b6ed0e,80.0,128.0,152500.0,258500.0,632.50,645.43,12.93
2,inf,0.0,48.0,0.0,92500.0,236.67,242.54,5.87
3,788b0a71,128.0,128.0,370000.0,370000.0,411.67,411.67,0.00
4,a22bdd7c,5.0,5.0,9500.0,9500.0,406.67,406.67,0.00
...,...,...,...,...,...,...,...,...
123,0eb86326,30.0,30.0,192000.0,192000.0,NaN,NaN,NaN
124,6342c79e,64.0,64.0,130000.0,130000.0,NaN,NaN,NaN
125,668c569c,64.0,64.0,130000.0,130000.0,NaN,NaN,NaN
126,6fdf4d0d,96.0,96.0,190000.0,190000.0,NaN,NaN,NaN


## Checking what went wrong with VM data 

In [9]:
import duckdb as ddb
con = ddb.connect(':memory:')

In [37]:
VM_HARDWARE = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/vms/2024-12-14T000000Z_2025-04-13T235959Z/vms_fixed.csv"
VM_DATA = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/**/*.csv"

In [38]:
con.execute(f"""CREATE OR REPLACE TABLE vmhardware AS SELECT * FROM '{VM_HARDWARE}'""")
con.execute(f"""
                CREATE OR REPLACE VIEW vm_data AS
                SELECT *
                FROM read_csv_auto('{VM_DATA}',
                                filename=false,
                                union_by_name=true)
            """)

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE vm_merged AS

    SELECT
        d.timestamp,
        d.vm_id,
        d.hypervisor_name,
        d.hypervisor_group,
        d.scaphandre_vm_power_total_watts,

        h.user_id,
        h.project_id,
        h.vcpus AS vm_cpu,
        h.memory_MB AS vm_memory_mb,
        h.root_GB,
        h.ephemeral_GB

    FROM vm_data d
    LEFT JOIN vmhardware h
    ON d.vm_id = h.vm_id;
    """)

,Count
0,15942293


In [40]:
con.query("""SELECT * FROM vm_merged LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬────────────┬────────┬──────────────┬─────────┬──────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ user_id  │ project_id │ vm_cpu │ vm_memory_mb │ root_gb │ ephemeral_gb │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │  varchar   │ double │    double    │ double  │    double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼────────────┼────────┼──────────────┼─────────┼──────────────┤
│ 2025-01-16 09:15:00+01   │ 3e322f7f │ 3eb76848        │ 11cdff15         │                            98.0 │ 0fb4c56f │ 43d3e415   │   48.0 │      94000.0 │    30.0 │        270.0 │
│ 2025-01-16 09:18:00+01   │ 3e322f7f │ 3eb76848        │ 11cdff15         │    

Amount of Nulls in scaphandre_vm_power total watts

In [41]:
con.query("""SELECT COUNT(*) 
FROM vm_merged 
WHERE scaphandre_vm_power_total_watts IS NULL""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       971896 │
└──────────────┘



Filling NULL variables

In [42]:
con.execute("""
CREATE OR REPLACE TABLE vm_filled AS

SELECT *,
    LAST_VALUE(scaphandre_vm_power_total_watts IGNORE NULLS)
    OVER (
        PARTITION BY vm_id
        ORDER BY timestamp
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS power_filled

FROM vm_merged;""").df()
    

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,15942293


Checking if power filled is null

In [43]:
con.query("""SELECT COUNT(*) 
FROM vm_filled 
WHERE power_filled IS NULL""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│            0 │
└──────────────┘



In [44]:
con.execute("""
CREATE OR REPLACE TABLE vm_final AS
    SELECT *,
        COALESCE(power_filled, 0) AS vm_power
    FROM vm_filled;""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Count
0,15942293


In [51]:
con.query("""SELECT * FROM vm_final WHERE vm_id = 'inf'""").show()

┌──────────────────────────┬─────────┬─────────────────┬──────────────────┬─────────────────────────────────┬─────────┬────────────┬────────┬──────────────┬─────────┬──────────────┬──────────────┬──────────┐
│        timestamp         │  vm_id  │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ user_id │ project_id │ vm_cpu │ vm_memory_mb │ root_gb │ ephemeral_gb │ power_filled │ vm_power │
│ timestamp with time zone │ varchar │     varchar     │     varchar      │             double              │ varchar │  varchar   │ double │    double    │ double  │    double    │    double    │  double  │
└──────────────────────────┴─────────┴─────────────────┴──────────────────┴─────────────────────────────────┴─────────┴────────────┴────────┴──────────────┴─────────┴──────────────┴──────────────┴──────────┘
                                                                                                    0 rows                                                              

In [53]:
con.query("""SELECT 
    vm_power,
    COUNT(*) as count,
    COUNT(DISTINCT vm_id) as distinct_vms
FROM vm_final
WHERE timestamp = '2025-02-12 11:57:00'  -- one of your test timestamps
GROUP BY vm_power
ORDER BY vm_power ASC
LIMIT 20""").show()


┌──────────┬───────┬──────────────┐
│ vm_power │ count │ distinct_vms │
│  double  │ int64 │    int64     │
├──────────┼───────┼──────────────┤
│      0.0 │    66 │           66 │
│     0.01 │    39 │           39 │
│     0.02 │    63 │           63 │
│     0.03 │    20 │           20 │
│     0.04 │    11 │           11 │
│     0.05 │    11 │           11 │
│     0.06 │    16 │           16 │
│     0.07 │     9 │            9 │
│     0.08 │     3 │            3 │
│     0.09 │    12 │           12 │
│      0.1 │     4 │            4 │
│     0.11 │     5 │            5 │
│     0.12 │     4 │            4 │
│     0.13 │     2 │            2 │
│     0.14 │     1 │            1 │
│     0.15 │     4 │            4 │
│     0.16 │     1 │            1 │
│     0.17 │     3 │            3 │
│     0.18 │     2 │            2 │
│     0.19 │     3 │            3 │
└──────────┴───────┴──────────────┘
  20 rows               3 columns



## Analysing Node Snapshot once again

In [54]:
con.execute(f"""CREATE OR REPLACE VIEW node_snapshot AS SELECT * FROM read_parquet('/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/node_snapshot.parquet')""")

In [57]:
inf_df = con.query("""SELECT * FROM node_snapshot WHERE node_name = 'inf'""").to_df()
print(inf_df.shape)
inf_df.head()

(55200, 15)


,timestamp,node_name,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,host_state,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2024-12-14 01:00:00+01:00,inf,54.47,285.33,48.0,1600.0,96570.382812,648.497396,normal,0,0.0,0.0,0.0,285.33,285.33
1,2024-12-14 01:03:00+01:00,inf,60.72,296.67,48.0,1600.0,96570.382812,640.805339,normal,0,0.0,0.0,0.0,296.67,296.67
2,2024-12-14 01:06:00+01:00,inf,61.53,298.00,48.0,1600.0,96570.382812,631.932292,normal,0,0.0,0.0,0.0,298.00,298.00
3,2024-12-14 01:09:00+01:00,inf,59.72,295.33,48.0,1600.0,96570.382812,638.922526,normal,0,0.0,0.0,0.0,295.33,295.33
4,2024-12-14 01:12:00+01:00,inf,52.44,283.33,48.0,1600.0,96570.382812,636.363932,normal,0,0.0,0.0,0.0,283.33,283.33


In [ ]:
eb5d8f42_df = con.query("""SELECT * FROM node_snapshot WHERE node_name = 'eb5d8f42' AND timestamp = '2025-02-15 12:00:00'""").to_df()
print(eb5d8f42_df.shape)
eb5d8f42_df.head()

(1, 15)


,timestamp,node_name,cpu_usage_percent,ipmi_system_power_watts,cpu_capacity,power_capacity,memory_capacity_mb,memory_available_mb,host_state,vm_count,vm_power_allocated,cpu_allocated,memory_allocated_mb,baseline_power,simulated_power
0,2025-02-15 12:00:00+01:00,eb5d8f42,NaN,NaN,96.0,2000.0,NaN,NaN,normal,0,0.0,0.0,0.0,NaN,NaN


In [ ]:
con.query("""-- Check for NULL/NaN in node_snapshot
SELECT node_name, COUNT(*) 
FROM node_snapshot
WHERE node_name IS NULL 
   OR node_name = 'inf'
GROUP BY node_name;""").show()

┌───────────┬──────────────┐
│ node_name │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ NULL      │         2880 │
│ inf       │        55200 │
└───────────┴──────────────┘



In [63]:
con.query("""SELECT MIN(timestamp), MAX(timestamp), COUNT(DISTINCT timestamp)
FROM node_snapshot;""").show()

┌──────────────────────────┬──────────────────────────┬─────────────────────────────┐
│     min("timestamp")     │     max("timestamp")     │ count(DISTINCT "timestamp") │
│ timestamp with time zone │ timestamp with time zone │            int64            │
├──────────────────────────┼──────────────────────────┼─────────────────────────────┤
│ 2024-12-14 01:00:00+01   │ 2025-04-14 01:57:00+02   │                       58080 │
└──────────────────────────┴──────────────────────────┴─────────────────────────────┘



In [64]:
con.query("""SELECT MIN(timestamp), MAX(timestamp), COUNT(DISTINCT timestamp)
FROM vm_final;""").show()

┌──────────────────────────┬──────────────────────────┬─────────────────────────────┐
│     min("timestamp")     │     max("timestamp")     │ count(DISTINCT "timestamp") │
│ timestamp with time zone │ timestamp with time zone │            int64            │
├──────────────────────────┼──────────────────────────┼─────────────────────────────┤
│ 2024-12-14 01:00:00+01   │ 2025-04-14 01:57:00+02   │                       58080 │
└──────────────────────────┴──────────────────────────┴─────────────────────────────┘



In [65]:
con.query("""SELECT * FROM vm_final;""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬────────────┬────────┬──────────────┬─────────┬──────────────┬──────────────┬──────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ user_id  │ project_id │ vm_cpu │ vm_memory_mb │ root_gb │ ephemeral_gb │ power_filled │ vm_power │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │  varchar   │ double │    double    │ double  │    double    │    double    │  double  │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼────────────┼────────┼──────────────┼─────────┼──────────────┼──────────────┼──────────┤
│ 2025-03-18 16:51:00+01   │ 6a455ac2 │ 1d247e53        │ a6177608         │                             0.0 │ 4a88a504 │ 4bd44b4c   │    2.0 │       4000.0 │  

In [67]:
con.query("""SELECT 
COUNT(DISTINCT timestamp) as unique_timestamps,
COUNT(DISTINCT vm_id) as unique_vms,
COUNT(DISTINCT hypervisor_name) as unique_nodes,
COUNT(DISTINCT hypervisor_group) as unique_groups,
FROM vm_final;""").show()

┌───────────────────┬────────────┬──────────────┬───────────────┐
│ unique_timestamps │ unique_vms │ unique_nodes │ unique_groups │
│       int64       │   int64    │    int64     │     int64     │
├───────────────────┼────────────┼──────────────┼───────────────┤
│             58080 │        612 │          117 │             6 │
└───────────────────┴────────────┴──────────────┴───────────────┘



In [68]:
con.query("""SELECT 
COUNT(DISTINCT timestamp) as unique_timestamps,
COUNT(DISTINCT node_name) as unique_nodes,
FROM node_snapshot;""").show()

┌───────────────────┬──────────────┐
│ unique_timestamps │ unique_nodes │
│       int64       │    int64     │
├───────────────────┼──────────────┤
│             58080 │          130 │
└───────────────────┴──────────────┘



## Maybe the problem starts from engineered features

In [69]:
NODES_FEATURES = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/engineered_features.parquet"
con.execute(f"""CREATE TABLE nodes_table AS SELECT * FROM read_parquet('{NODES_FEATURES}')""")
con.query(f"""SELECT * from nodes_table LIMIT 5""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────┬────────────┬────────────────────┬───────────────────┬───────────────────┬─────────────────────────┬─────────────────────────────────┬──────────────────────────────┬──────────────────────────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┐
│        timestamp         │ node_name │ node_group │ memory_total_bytes │ memory_used_bytes │ memory_free_bytes │ ipmi_system_power_watts │ ipmi_system_power_watts_imputed │ scaphandre_power_total_watts │ scaphandre_power_total_watts_imputed │ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_per

In [70]:
con.query("""SELECT 
COUNT(DISTINCT timestamp) as unique_timestamps,
COUNT(DISTINCT node_name) as unique_nodes,
FROM nodes_table;""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬──────────────┐
│ unique_timestamps │ unique_nodes │
│       int64       │    int64     │
├───────────────────┼──────────────┤
│             58080 │          130 │
└───────────────────┴──────────────┘



In [71]:
con.query("""
SELECT node_name, COUNT(*) 
FROM nodes_table
WHERE node_name IS NULL 
   OR node_name = 'inf'
GROUP BY node_name;""").show()

┌───────────┬──────────────┐
│ node_name │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ NULL      │       365760 │
│ inf       │        55200 │
└───────────┴──────────────┘



## New Engineered Features

In [3]:
NODES_FEATURES = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/engineered_features_new.parquet"
con.execute(f"""CREATE TABLE features_table AS SELECT * FROM read_parquet('{NODES_FEATURES}')""")
con.query(f"""SELECT * from features_table LIMIT 5""").show()

┌──────────────────────────┬───────────┬────────────┬────────────────────┬───────────────────┬───────────────────┬─────────────────────────┬──────────────────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬──────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┐
│        timestamp         │ node_name │ node_group │ memory_total_bytes │ memory_used_bytes │ memory_free_bytes │ ipmi_system_power_watts │ scaphandre_power_total_watts │ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_percent │ cpu_interrupt_percent │ load_shortterm_percent │ load_midterm_percent │ load_longterm_percent │  memory_util_ratio   │   disk_util_ratio  

In [4]:
con.query("""SELECT 
COUNT(DISTINCT timestamp) as unique_timestamps,
COUNT(DISTINCT node_name) as unique_nodes,
FROM features_table;""").show()

┌───────────────────┬──────────────┐
│ unique_timestamps │ unique_nodes │
│       int64       │    int64     │
├───────────────────┼──────────────┤
│             58080 │          130 │
└───────────────────┴──────────────┘



In [5]:
con.query("""
SELECT node_name, COUNT(*) 
FROM features_table
WHERE node_name IS NULL 
   OR node_name = 'inf'
GROUP BY node_name;""").show()

┌───────────┬──────────────┐
│ node_name │ count_star() │
│  varchar  │    int64     │
└───────────┴──────────────┘
           0 rows         



In [11]:
Failed_Placements = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/failed_placements_example.parquet"
con.execute(f"""CREATE TABLE failed_placements AS SELECT * FROM read_parquet('{Failed_Placements}')""")
con.query(f"""SELECT * from failed_placements LIMIT 5""").show()

┌──────────┬─────────────┬──────────┬────────┬──────────────┬────────────────────────┬───────────────────────────┬──────────────────────────┐
│  vm_id   │ source_node │ vm_power │ vm_cpu │ vm_memory_mb │ insufficient_cpu_hosts │ insufficient_memory_hosts │ insufficient_power_hosts │
│ varchar  │   varchar   │  double  │ double │    double    │         int64          │           int64           │          int64           │
├──────────┼─────────────┼──────────┼────────┼──────────────┼────────────────────────┼───────────────────────────┼──────────────────────────┤
│ b3382983 │ 5433c3e6    │   485.48 │  128.0 │     370000.0 │                     31 │                        31 │                        8 │
│ 5d14c1f6 │ cbf4a974    │    21.45 │  128.0 │     370000.0 │                     31 │                        31 │                        0 │
│ 3b16a26a │ 85c6cd05    │     5.61 │   16.0 │      46000.0 │                     31 │                        28 │                        0 │
│ ca09

In [10]:
placements = "/Users/biancachiusano/Desktop/uva/thesis/energy-aware-vm-consolidation/datasets/cloud_energy_consumption/processed/placements_example.parquet"
con.execute(f"""CREATE TABLE placements AS SELECT * FROM read_parquet('{placements}')""")
con.query(f"""SELECT * from placements LIMIT 5""").show()

┌──────────┬─────────────┬─────────────┬──────────┬────────┬──────────────┐
│  vm_id   │ source_node │ target_node │ vm_power │ vm_cpu │ vm_memory_mb │
│ varchar  │   varchar   │   varchar   │  double  │ double │    double    │
├──────────┼─────────────┼─────────────┼──────────┼────────┼──────────────┤
│ aa7cd1e3 │ 85ebbabd    │ e2b6ed0e    │     8.92 │   32.0 │      60000.0 │
│ efe7ef34 │ 46362b76    │ 1e64c783    │     7.34 │   64.0 │     130000.0 │
│ 6a0c4f32 │ 22655375    │ e2b6ed0e    │     6.95 │    8.0 │      23000.0 │
│ b0036066 │ 28e8b849    │ e2b6ed0e    │     4.36 │    4.0 │      11000.0 │
│ 2d329ea1 │ 58512373    │ e2b6ed0e    │     2.25 │    4.0 │       7500.0 │
└──────────┴─────────────┴─────────────┴──────────┴────────┴──────────────┘

